# 02 Interpret Results

Load a completed sweep and regenerate figures + robust summaries.

This notebook now writes the shared reporting artifacts, then shows the current trial scorecard and previous same-env run before the figure gallery.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("Project root:", PROJECT_ROOT)


In [ ]:
import pandas as pd

from ess_ope.evaluation.reporting import build_trial_index, generate_run_artifacts

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 220)

latest_dir = (PROJECT_ROOT / "results/latest").resolve()
pq = latest_dir / "sweep_results.parquet"
csv = latest_dir / "sweep_results.csv"

if pq.exists():
    df = pd.read_parquet(pq)
    source = pq
elif csv.exists():
    df = pd.read_csv(csv)
    source = csv
else:
    raise FileNotFoundError("No sweep_results in results/latest")

print("Loaded:", source)
print("Rows:", len(df))
if "env_name" in df.columns:
    print("env_name:", sorted(df.env_name.unique().tolist()))
print("alphas:", sorted(df.alpha.unique().tolist()))
print("betas:", sorted(df.beta.unique().tolist()))
print("K:", sorted(df.K.unique().tolist()))
if "transition_strength" in df.columns:
    print("transition_strengths:", sorted(df.transition_strength.unique().tolist()))
if "reward_mean_scale" in df.columns:
    print("reward_mean_scales:", sorted(df.reward_mean_scale.unique().tolist()))
if "reward_std" in df.columns:
    print("reward_stds:", sorted(df.reward_std.unique().tolist()))
if "chain_variant" in df.columns:
    print("chain_variants:", sorted(df.chain_variant.unique().tolist()))


In [ ]:
fig_dir = latest_dir / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

artifacts = generate_run_artifacts(
    df,
    output_dir=fig_dir,
    fixed_alpha=None,
    fixed_beta=0.0,
    bootstrap_samples=16000,
    bootstrap_max_points=200000,
    progress=True,
)
benchmark_report = artifacts["benchmark_report"]
trial_scorecard = artifacts["trial_scorecard"]
paper_claims = artifacts["paper_claims"]
paper_claim_summary = artifacts["paper_claim_summary"]
chain_bandit_sensitivity = artifacts["chain_bandit_sensitivity_summary"]
trial_index = build_trial_index(PROJECT_ROOT / "results")
current_run = latest_dir.name
current_env = df["env_name"].iloc[0] if "env_name" in df.columns else "random_mdp"
previous_same_env = trial_index[(trial_index["env_name"] == current_env) & (trial_index["run_id"] < current_run)].tail(1)

print("Saved to:", fig_dir)
trial_scorecard


In [ ]:
print("Current run:", current_run)
print("Current env:", current_env)
trial_scorecard


In [ ]:
if not previous_same_env.empty:
    previous_same_env
else:
    print("No previous same-env run found in results/trial_index.csv")


In [ ]:
if not chain_bandit_sensitivity.empty:
    chain_bandit_sensitivity
else:
    paper_claims


In [ ]:
paper_claim_summary


In [ ]:
from IPython.display import Image, display

for name in [
    "benchmark_fig1_ess_vs_error_by_estimator.png",
    "benchmark_fig2_same_ess_different_error.png",
    "benchmark_fig3_ess_changes_error_stability.png",
    "benchmark_fig4_fan_estimate_vs_ess.png",
    "benchmark_fig5_mean_variance_sensitivity.png",
]:
    p = fig_dir / name
    if p.exists():
        print(name)
        display(Image(filename=str(p)))
